In [0]:
from pyspark.sql import functions as F
import pandas as pd
from pyspark.sql.window import Window
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

## Initialize the notebook
MOM_month = "2026-02"
snapshot_date = '20260228'
var_visit_year = [2019,2025]


from pyspark.sql import SparkSession
def create_spark_session(app_name: str = "MOM") -> SparkSession:

    return (
        SparkSession.builder
        .appName(app_name)
        .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
        .getOrCreate()
    )



T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_sessions = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_ticket = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_genre = spark.table("cpx_dataanalytics_gold.customer360.film_composite_genres")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")
T_emails = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")
T_demo = spark.table('cpx_dataanalytics_gold.customer360.cpx_d_customer')

In [0]:
pip install openpyxl

In [0]:
pdf = pd.read_excel(
    "/Workspace/Users/ranjan.mishra@cineplex.com/Codebase/CineClub/Monthly Reports/February 2026/Subscriptions - FEB 26.xlsx",
    usecols=[
        "customers.id",
        "subscriptions.id",
        "subscriptions.plan_id",
        "subscriptions.status",
        "subscriptions.current_term_start",
        "subscriptions.current_term_end",
        "subscriptions.created_at",
        "subscriptions.cancelled_at",
    ],
)

# 3) Parse datetime columns in pandas and convert to a consistent string format (Arrow-safe)
dt_cols = [
    "subscriptions.current_term_start",
    "subscriptions.current_term_end",
    "subscriptions.created_at",
    "subscriptions.cancelled_at",
]

for c in dt_cols:
    pdf[c] = pd.to_datetime(pdf[c], format="%d-%b-%Y %H:%M", errors="coerce")
    pdf[c] = pdf[c].dt.strftime("%Y-%m-%d %H:%M:%S")  # NaT -> NaN

# 4) Ensure ID/status fields are strings (prevents mixed-type Arrow issues)
for c in ["customers.id", "subscriptions.id", "subscriptions.plan_id", "subscriptions.status"]:
    pdf[c] = pdf[c].astype("string")


# 5) Convert pandas -> Spark DataFrame (remote cluster)
df = spark.createDataFrame(pdf)

# 6) Project ONLY the final desired columns (no repetition):
#    - Rename dotted columns to clean names
#    - Parse the datetime strings into Spark timestamps
df_main = (
    df
    .select(
        F.col("`customers.id`").alias("customer_id"),
        F.col("`subscriptions.id`").alias("subscription_id"),
        F.col("`subscriptions.plan_id`").alias("plan_id"),
        F.col("`subscriptions.status`").alias("subscription_status"),
        F.to_timestamp(F.col("`subscriptions.current_term_start`"), "yyyy-MM-dd HH:mm:ss").alias("current_term_start"),
        F.to_timestamp(F.col("`subscriptions.current_term_end`"),   "yyyy-MM-dd HH:mm:ss").alias("current_term_end"),
        F.to_timestamp(F.col("`subscriptions.created_at`"),         "yyyy-MM-dd HH:mm:ss").alias("created_at"),
        F.to_timestamp(F.col("`subscriptions.cancelled_at`"),       "yyyy-MM-dd HH:mm:ss").alias("cancelled_at"),
    )
    .dropDuplicates([
        "customer_id",
        "plan_id",
        "subscription_status",
        "current_term_start",
        "current_term_end",
        "created_at",
        "cancelled_at"
    ])
    .withColumn("created_at_year_month", F.date_format(F.col("created_at"), "yyyy-MM"))
    .withColumn("current_term_end_year_month", F.date_format(F.col("current_term_end"), "yyyy-MM"))
    .withColumn("cancelled_at_year_month", F.date_format(F.col("cancelled_at"), "yyyy-MM"))
    .withColumn(
        "days_in_program",
        F.when(
            F.col("cancelled_at").isNull() | F.col("created_at").isNull(),
            F.lit(-9999)
        ).otherwise(F.datediff(F.col("cancelled_at"), F.col("created_at"))).cast("int")
    )
)

# Overview

In [0]:
signups = df_main.filter(F.col("created_at_year_month") == MOM_month)

net_new = (
    df_main
    # keep only Active / Non Renewing created in the target month
    .filter(F.col("subscription_status").isin("Active", "Non Renewing"))
    .filter(F.col("created_at_year_month") == MOM_month)
    # retain only customers who have exactly 1 record in the full dataset
    .join(
        df_main
        .groupBy("customer_id")
        .count()
        .filter(F.col("count") == 1)
        .select("customer_id"),
        on="customer_id",
        how="inner"
    )
)

print("Signups:", signups.count())
print("Net New:", net_new.select("customer_id").distinct().count())

In [0]:
total_exits = df_main.filter(F.col("cancelled_at_year_month") == MOM_month)
print("Total Exits:", total_exits.count())

tmp = df_main.filter(F.col("subscription_status").isin("Active", "Non Renewing"))
current_annual = tmp.filter(F.col("plan_id") == "annual1")
# merged: tmp step folded into current_annual
upgrade_to_annual = (
    total_exits.select(F.col("customer_id")).distinct()
    .join(
        current_annual.select(F.col("customer_id")).distinct(),
        on="customer_id",
        how="inner"
    )
)

net_exits = (
    total_exits.select("customer_id").distinct().count()
    - upgrade_to_annual.select("customer_id").distinct().count()
)

print("Upgrade to Annual:", upgrade_to_annual.count())
print("Current Annual:", current_annual.count())

current_annual_exits = current_annual.join(
    total_exits.select("customer_id").distinct(),
    on="customer_id",
    how="inner"
)

print("Net Exits:", net_exits)


In [0]:
ending_balance_active = (
    df_main
    .filter(F.col("subscription_status").isin("Active", "Non Renewing"))
    .count()
)

w = Window.partitionBy()

status_distribution = (
    df_main
    .filter(F.col("subscription_status").isin("Active", "Non Renewing"))
    .groupBy("subscription_status")
    .agg(F.count("*").alias("cnt"))
    .withColumn(
        "proportion",
        F.col("cnt") / F.sum("cnt").over(w)
    )
)

status_distribution.display()

print("Ending Balance of Active Members:", ending_balance_active)

In [0]:
plan_distribution = (
    df_main
    .filter(F.col("subscription_status").isin("Active", "Non Renewing"))
    .groupBy("plan_id")
    .agg(F.count("*").alias("cnt"))
    .withColumn(
        "proportion",
        F.round(F.col("cnt") / F.sum("cnt").over(w),4)
    )
)

plan_distribution.display()

# Subscriptions

In [0]:
monthly_signup = (
    df_main
    .filter(F.col("plan_id") == "monthly1")
    .filter(F.col("created_at_year_month") == MOM_month)
)

annual_signup = (
    df_main
    .filter(F.col("plan_id") == "annual1")
    .filter(F.col("created_at_year_month") == MOM_month)
)

gift_plan_signup = (
    df_main
    .filter(F.col("plan_id") == "gift-plan")
    .filter(F.col("created_at_year_month") == MOM_month)
)

print(f"Total till Date Signups:", df_main.dropDuplicates(["customer_id"]).count())
print("Monthly Signups:", monthly_signup.count())
print("Annual Signups:", annual_signup.count())
print("Gift-Plan Signups:", gift_plan_signup.count())

In [0]:
prev_max_created = (
    df_main
    .filter(F.col("created_at_year_month") < MOM_month)
    .groupBy("customer_id")
    .agg(F.max("created_at").alias("created_at"))
)

previous_signup_df_ = (
    prev_max_created
    .join(df_main, on=["customer_id", "created_at"], how="inner")
    .dropDuplicates(["customer_id", "created_at"])
)

# reuse this once everywhere
previous_signup_list = previous_signup_df_.select("customer_id").distinct()

# First Time Monthly
first_time_df = monthly_signup.join(previous_signup_list, on="customer_id", how="left_anti")
monthly_signup_count = monthly_signup.count()
first_time_count = first_time_df.count()

print("% Monthly First Time:", "{:.2%}".format(first_time_count / monthly_signup_count if monthly_signup_count > 0 else 0))
print("# of Monthly First Time:", "{:,d}".format(first_time_count))

# First Time Annual
annual_first_time_df = annual_signup.join(previous_signup_list, on="customer_id", how="left_anti")
annual_signup_count = annual_signup.count()
annual_first_time_count = annual_first_time_df.count()

print("% Annual First Time:", "{:.2%}".format(annual_first_time_count / annual_signup_count if annual_signup_count > 0 else 0))
print("# of Annual First Time:", "{:,d}".format(annual_first_time_count))

# First Time Gift-Plan
gift_plan_first_time_df = gift_plan_signup.join(previous_signup_list, on="customer_id", how="left_anti")
gift_plan_signup_count = gift_plan_signup.count()
gift_plan_first_time_count = gift_plan_first_time_df.count()

print("% Gift-Plan First Time:", "{:.2%}".format(gift_plan_first_time_count / gift_plan_signup_count if gift_plan_signup_count > 0 else 0))
print("# of Gift-Plan First Time:", "{:,d}".format(gift_plan_first_time_count))


In [0]:


# -------------------------------------------------------------------
# Shared: prior customer list (used across Monthly / Annual / Gift-Plan)
# -------------------------------------------------------------------
previous_signup_customer_ids = previous_signup_df_.select("customer_id").distinct()

# -------------------------------------------------------------------
# Split by previous plan type (same variables you already use)
# -------------------------------------------------------------------
previous_monthly_signup = previous_signup_df_.filter(F.col("plan_id") == "monthly1")
previous_annual_signup  = previous_signup_df_.filter(F.col("plan_id") == "annual1")
previous_gift_signup    = previous_signup_df_.filter(F.col("plan_id") == "gift-plan")

# For each plan subset, get max(created_at) per customer_id (kept intentionally)
previous_monthly_signup_df = (
    previous_monthly_signup
    .groupBy("customer_id")
    .agg(F.max("created_at").alias("max_created_at"))
)

previous_annual_signup_df = (
    previous_annual_signup
    .groupBy("customer_id")
    .agg(F.max("created_at").alias("max_created_at"))
)

previous_gift_plan_signup_df = (
    previous_gift_signup
    .groupBy("customer_id")
    .agg(F.max("created_at").alias("max_created_at"))
)

# Customer_id lists for joins (kept aligned with your original logic)
previous_monthly_customer_ids = previous_monthly_signup_df.select("customer_id").distinct()
previous_annual_customer_ids  = previous_annual_signup.select("customer_id").distinct()
previous_gift_customer_ids    = previous_gift_signup.select("customer_id").distinct()

# =====================================================================================
# MONTHLY RETURNING
# =====================================================================================
return_df = monthly_signup.join(previous_signup_customer_ids, on="customer_id", how="inner")

monthly_signup_count = monthly_signup.count()
return_count = return_df.count()
display(Markdown("**% Monthly Returning**"))
print("% Monthly Returning:", "{:.2%}".format(return_count / monthly_signup_count if monthly_signup_count > 0 else 0))
print("# of Monthly Returning:", "{:,d} \n".format(return_count))

return_from_monthly_df = monthly_signup.join(previous_monthly_customer_ids, on="customer_id", how="inner")
return_from_monthly_cnt = return_from_monthly_df.count()
print("% Returning (from Monthly):", "{:.2%}".format(return_from_monthly_cnt / return_count if return_count > 0 else 0))
print("Returning (from Monthly):", "{:,d} \n".format(return_from_monthly_cnt))

return_from_annual_df = monthly_signup.join(previous_annual_customer_ids, on="customer_id", how="inner")
return_from_annual_cnt = return_from_annual_df.count()
print("% Returning (from Annual):", "{:.2%}".format(return_from_annual_cnt / return_count if return_count > 0 else 0))
print("Returning (from Annual):", "{:,d} \n".format(return_from_annual_cnt))

return_from_gift_plan_df = monthly_signup.join(previous_gift_customer_ids, on="customer_id", how="inner")
return_from_gift_plan_cnt = return_from_gift_plan_df.count()
print("% Returning (from Gift-Plan):", "{:.2%}".format(return_from_gift_plan_cnt / return_count if return_count > 0 else 0))
print("Returning (from Gift-Plan):", "{:,d} \n".format(return_from_gift_plan_cnt))

# =====================================================================================
# ANNUAL RETURNING
# =====================================================================================
annual_return_df = annual_signup.join(previous_signup_customer_ids, on="customer_id", how="inner")

annual_signup_count = annual_signup.count()
annual_return_count = annual_return_df.count()
display(Markdown("**% Annual Returning**"))
print("% Annual Returning:", "{:.2%}".format(annual_return_count / annual_signup_count if annual_signup_count > 0 else 0))
print("# of Annual Returning:", "{:,d} \n".format(annual_return_count))

denom_annual_return = annual_return_count  # same semantics as annual_return_df.count()

return_from_monthly_df = annual_signup.join(previous_monthly_customer_ids, on="customer_id", how="inner")
return_from_monthly_cnt = return_from_monthly_df.count()
print("% Returning (from Monthly):", "{:.2%}".format(return_from_monthly_cnt / denom_annual_return if denom_annual_return > 0 else 0))
print("Returning (from Monthly):", "{:,d} \n".format(return_from_monthly_cnt))

return_from_annual_df = annual_signup.join(previous_annual_customer_ids, on="customer_id", how="inner")
return_from_annual_cnt = return_from_annual_df.count()
print("% Returning (from Annual):", "{:.2%}".format(return_from_annual_cnt / denom_annual_return if denom_annual_return > 0 else 0))
print("Returning (from Annual):", "{:,d} \n".format(return_from_annual_cnt))

return_from_gift_plan_df = annual_signup.join(previous_gift_customer_ids, on="customer_id", how="inner")
return_from_gift_plan_cnt = return_from_gift_plan_df.count()
print("% Returning (from Gift-Plan):", "{:.2%}".format(return_from_gift_plan_cnt / denom_annual_return if denom_annual_return > 0 else 0))
print("Returning (from Gift-Plan):", "{:,d} \n".format(return_from_gift_plan_cnt))

# =====================================================================================
# GIFT-PLAN RETURNING
# =====================================================================================
gift_plan_return_df = gift_plan_signup.join(previous_signup_customer_ids, on="customer_id", how="inner")

gift_plan_signup_count = gift_plan_signup.count()
gift_plan_return_count = gift_plan_return_df.count()
display(Markdown("**% Gift-Plan Returning**"))
print("% Gift-Plan Returning(Gift-Plan):", "{:.2%}".format(gift_plan_return_count / gift_plan_signup_count if gift_plan_signup_count > 0 else 0))
print("# of Gift-Plan Returning(Gift-Plan):", "{:,d} \n".format(gift_plan_return_count))

denom_gift_return = gift_plan_return_count  # same semantics as gift_plan_return_df.count()

return_from_monthly_df = gift_plan_signup.join(previous_monthly_customer_ids, on="customer_id", how="inner")
return_from_monthly_cnt = return_from_monthly_df.count()
print("% Returning (from Monthly)(Gift-Plan):", "{:.2%}".format(return_from_monthly_cnt / denom_gift_return if denom_gift_return > 0 else 0))
print("Returning (from Monthly)(Gift-Plan):", "{:,d} \n".format(return_from_monthly_cnt))

return_from_annual_df = gift_plan_signup.join(previous_annual_customer_ids, on="customer_id", how="inner")
return_from_annual_cnt = return_from_annual_df.count()
print("% Returning (from Annual)(Gift-Plan):", "{:.2%}".format(return_from_annual_cnt / denom_gift_return if denom_gift_return > 0 else 0))
print("Returning (from Annual)(Gift-Plan):", "{:,d} \n".format(return_from_annual_cnt))

return_from_gift_plan_df = gift_plan_signup.join(previous_gift_customer_ids, on="customer_id", how="inner")
return_from_gift_plan_cnt = return_from_gift_plan_df.count()
print("% Returning (from Gift-Plan)(Gift-Plan):", "{:.2%}".format(return_from_gift_plan_cnt / denom_gift_return if denom_gift_return > 0 else 0))
print("Returning (from Gift-Plan)(Gift-Plan):", "{:,d} \n".format(return_from_gift_plan_cnt))

# Demographics and Visitation

In [0]:
# =========================
# BLOCK 1 — MAIN DATA INITIALIZATION
# (df_demographics_main, df_trans_main, ptd_signup)
# =========================
# Demographics snapshot
df_demographics_main = (
    T_subscriber.join(T_demo, "cde_id", "left")
    .filter(F.col("snapshot_date_id") == snapshot_date)
    .select(
        "cde_id",
        "plan_id",
        "renewal_status",
        "date_of_birth_masked",
        "gender_text",
        "province",
        T_subscriber.cc_user_profile_id
    )
    .dropDuplicates(["cde_id"])
)

# Transactions (visits) main
w_cust = Window.partitionBy("cde_id")

df_trans_main = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre") &
        (T_trans.CDE_ID.isNotNull()) &
        (T_trans.SCENEFLAG == "Yes") &
        (T_trans.RecRoomFLAG == 'No') &
        (T_trans.STOREFLAG == "No")
    )
    .withColumnRenamed("CDE_ID", "cde_id")
    .groupBy("cde_id", "datefull", T_trans.LocationId)
    .agg(F.sum(F.col("Quantity")).alias("Net_Quantity"))
    .filter((F.col("Net_Quantity") > 0) & (F.col("cde_id") > 0))
    .withColumn("Total_Visits", F.count(F.lit(1)).over(w_cust))
)

# Program-to-date signup base (PTD) joined to demographics
ptd_signup = (
    df_main.select("customer_id", "days_in_program").alias("m")
    .join(
        T_demo.select(
            F.col("cc_user_profile_id").alias("cc_user_profile_id"),
            "gender_text",
            "date_of_birth_masked",
            "province",
            "cde_id"
        ).alias("d"),
        on=F.col("m.customer_id").cast("string") == F.col("d.cc_user_profile_id").cast("string"),
        how="inner"
    )
    .withColumn("age", F.year(F.current_date()) - F.col("date_of_birth_masked").cast("int"))
    .dropDuplicates(["customer_id"])
)

df_scene_member = (
     T_trans
     .filter(F.col("SCENEFLAG") == "Yes")
     .groupBy("cde_id")
     .agg(F.sum(F.col("Quantity")).alias("Net_Quantity"))
)

df_scene_demo =(
    df_scene_member
    .join(T_demo.select(
            F.col("cc_user_profile_id").alias("cc_user_profile_id"),
            "gender_text",
            "date_of_birth_masked",
            "province",
            "cde_id"
        ), "cde_id", "inner")
    .withColumn("age", F.year(F.current_date()) - F.col("date_of_birth_masked").cast("int"))
    .dropDuplicates(["cde_id"])
)

df_scene_member_visit = (
    T_trans
    .filter(F.col("SCENEFLAG") == "Yes")
    .filter(F.col("RecRoomFLAG")== "No")
    .filter(F.col("STOREFLAG")== "No")
    .filter(F.col("VisitDateId").like("2026%"))
    .groupBy("cde_id")
    .agg(F.sum(F.col("Quantity")).alias("Net_Quantity"))
)

df_cc_member_visit = (
    T_trans
    .filter(F.col("CineClubFLAG") == "Yes")
    .filter(F.col("VisitDateId").like("202602%"))
    .groupby("cde_id")
    .agg(F.sum(F.col("Quantity")).alias("Net_Quantity"))
)

df_ptd_member_visit = (
    T_trans
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(F.col("datefull").between("2021-01-01", "2026-02-28"))
    .filter(F.col("SCENEFLAG") == "Yes")
    .filter(F.col("RecRoomFLAG")== "No")
    .filter(F.col("STOREFLAG")== "No")
    .groupby("cde_id")
    .agg(F.sum(F.col("Quantity")).alias("Net_Quantity"))
)



In [0]:
df_main.filter(F.col("created_at_year_month") == MOM_month).display()

In [0]:
w = Window.partitionBy()

def show_gender_distribution(df_in, title, total_label):
    display(Markdown(f"**{title}**"))
    print(total_label, df_in.count())

    (
        df_in
        .withColumn("gender_clean", F.trim(F.col("gender_text")))
        .filter(F.col("gender_clean").isin("Male", "Female"))
        .groupBy("gender_clean")
        .agg(F.count("*").alias("cnt"))
        .withColumn("proportion", F.round(F.col("cnt") / F.sum("cnt").over(w),4))
        .orderBy("gender_clean")
        .display()
    )

# -------------------------
# (A) Month signups
# -------------------------
merge_df = (
    df_main
    .filter(F.col("created_at_year_month") == MOM_month)
    .dropDuplicates(["customer_id"])
    .alias("s")
    .join(
        df_demographics_main.alias("d"),
        F.col("s.customer_id") == F.col("d.cc_user_profile_id"),
        "left"
    )
)

show_gender_distribution(
    merge_df,
    "% Gender Distribution (Month)",
    "Signups:"
)

# -------------------------
# (B) Program till date (PTD)
# -------------------------
show_gender_distribution(
    ptd_signup,
    "% Gender Distribution (PTD)",
    "Total till Date Signups:"
)

show_gender_distribution(
    df_scene_demo, 
    "% Gender Distribution (SCENE)", 
    "Total till Date Scene Members:")

In [0]:
from pyspark.sql import functions as F, Window

w_all = Window.partitionBy()

def show_age_distribution(df_in, title, total_label, dob_col="date_of_birth_masked"):
    display(Markdown(f"**{title}**"))
    total_rows = df_in.count()
    print(total_label, total_rows)

    age_years = F.year(F.current_date()) - F.col(dob_col).cast("int")

    age_bin_expr = (
        F.when(age_years >= 65, "65+")
         .when(age_years >= 50, "50-64")
         .when(age_years >= 35, "35-49")
         .when(age_years >= 25, "25-34")
         .when(age_years >= 19, "19-24")
         .when(age_years >= 14, "14-18")
         .otherwise("NULL")
    )

    df_bins = (
        df_in
        .withColumn("age_years", age_years)
        .withColumn("age_bin", age_bin_expr)
        .groupBy("age_bin")
        .agg(F.count("*").alias("count"))
        # denominator excludes NULL
        .withColumn(
            "valid_total",
            F.sum(
                F.when(F.col("age_bin") != "NULL", F.col("count")).otherwise(0)
            ).over(w_all)
        )
        .withColumn(
            "proportion_pct",
            F.when(
                F.col("age_bin") != "NULL",
                F.round(F.col("count") / F.col("valid_total"), 4)
            )
        )
        .orderBy(
            F.when(F.col("age_bin") == "14-18", 1)
             .when(F.col("age_bin") == "19-24", 2)
             .when(F.col("age_bin") == "25-34", 3)
             .when(F.col("age_bin") == "35-49", 4)
             .when(F.col("age_bin") == "50-64", 5)
             .when(F.col("age_bin") == "65+", 6)
             .otherwise(99)
        )
    )

    df_bins.select("age_bin", "count", "proportion_pct").display()


# -------------------------
# (A) AGE Distribution (Month Signups)
# -------------------------
show_age_distribution(
    merge_df,
    "% Age Distribution (Month Signups)",
    "Signups:"
)

# -------------------------
# (B) AGE Distribution (Program Till Date - PTD)
# -------------------------
show_age_distribution(
    ptd_signup,
    "% Age Distribution (PTD)",
    "Total till Date Signups:"
)

show_age_distribution(
    df_scene_demo,
    "% Age Distribution (SCENE)",
    "Total till Date Scene Members:"
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from IPython.display import Markdown, display

w = Window.partitionBy()  # global window

def show_province_distribution(df_in, title, total_label, province_col="province"):
    display(Markdown(f"**{title}**"))

    # Provinces to collapse into ATL
    atl_provinces = ["NB", "NL", "NS", "PE", "PEI"]

    # Provinces / states / invalid codes to exclude (set to NULL)
    invalid_provinces = [
        "NT", "YT", "NU", "WRONG",
        "OH", "FL", "WA", "MA", "GA", "AZ", "NC", "KS",
        "CA", "MD", "NJ", "AL", "MI", "VA", "OR", "TX",
        "NY", "IL", "SC", "MN", "VT", "UT", "NV", "MT",
        "PA", "AR", "KY", "CT", "TN", "IA", "DC", "OK", "NH", "CO",
        "WV", "IN", "DE", "LA", "ME", "ND"
    ]

    df_norm = (
        df_in
        .withColumn(
            "Province",
            F.when(F.upper(F.trim(F.col(province_col))).isin(atl_provinces), "ATL")
             .when(F.upper(F.trim(F.col(province_col))).isin(invalid_provinces), None)
             .otherwise(F.upper(F.trim(F.col(province_col))))
        )
    )

    # Exclude NULL / blank provinces
    df_non_null = df_norm.filter(F.col("Province").isNotNull() & (F.col("Province") != ""))

    # Build counts + proportions
    df_counts = (
        df_non_null
        .groupBy("Province")
        .agg(F.count("*").alias("count"))
        .withColumn("proportion", F.round(F.col("count") / F.sum("count").over(w), 4))
    )

    # Total (aligned to denominator used in proportions)
    total = df_non_null.count()
    print(total_label, total)

    # Custom display order
    order_list = ["ON", "QC", "BC", "AB", "MB", "SK", "ATL"]
    order_arr  = F.array(*[F.lit(x) for x in order_list])

    df_out = (
        df_counts
        .withColumn("sort_key", F.array_position(order_arr, F.col("Province")))
        .withColumn("sort_key", F.when(F.col("sort_key") == 0, 999).otherwise(F.col("sort_key")))
        .orderBy("sort_key")
        .drop("sort_key")
    )

    df_out.display()


# -------------------------
# (A) Province Distribution (Month Signups)
# -------------------------
show_province_distribution(
    merge_df,
    "% Province Distribution (Month Signups)",
    "Signups:",
    province_col="province"
)

# -------------------------
# (B) Province Distribution (Program Till Date - PTD)
# -------------------------
show_province_distribution(
    ptd_signup,
    "% Province Distribution (PTD)",
    "Total till Date Signups:",
    province_col="province"
)

# -------------------------
# (C) Province Distribution (SCENE)
# -------------------------
show_province_distribution(
    df_scene_demo,
    "% Province Distribution (SCENE)",
    "Total till Date Scene Members:",
    province_col="province"
)


In [0]:
def show_visit_distribution(df_cohort, cohort_name, id_col="cde_id"):
    display(Markdown(f"**Visit Distribution ({cohort_name})**"))

    # 1) Years in scope (restricted to var_visit_year)
    df_years = (
        df_trans_main
        .select(F.year(F.col("datefull")).alias("visit_year"))
        .filter(F.col("visit_year").isin(var_visit_year))
        .distinct()
    )

    # 2) Cohort population (distinct IDs)
    df_population = (
        df_cohort
        .select(F.col(id_col).alias("cde_id"))
        .filter(F.col(id_col).isNotNull())
        .distinct()
    )

    # 3) Visits per customer per year (restricted to cohort)
    df_visits_by_year = (
        df_trans_main
        .filter(F.col("cde_id").isNotNull())
        .withColumn("visit_year", F.year("datefull"))
        .filter(F.col("visit_year").isin(var_visit_year))
        .join(df_population, "cde_id", "inner")
        .groupBy("cde_id", "visit_year")
        .agg(F.count(F.lit(1)).alias("total_visits_year"))
    )

    # 4) Customer-year grid (explicit 0-visit years)
    df_customer_year = df_population.crossJoin(df_years)

    df_visits_full = (
        df_customer_year
        .join(df_visits_by_year, ["cde_id", "visit_year"], "left")
        .withColumn("total_visits_year", F.coalesce(F.col("total_visits_year"), F.lit(0)))
    )

    # 5) Bin annual visit counts (includes 0 Visits)
    df_visits_binned = (
        df_visits_full
        .withColumn(
            "visit_bin",
            F.when(F.col("total_visits_year") == 0, "0 Visits")
             .when(F.col("total_visits_year") == 1, "1 Visit")
             .when((F.col("total_visits_year") >= 2) & (F.col("total_visits_year") <= 3), "2-3 Visits")
             .when((F.col("total_visits_year") >= 4) & (F.col("total_visits_year") <= 6), "4-6 Visits")
             .when((F.col("total_visits_year") >= 7) & (F.col("total_visits_year") <= 12), "7-12 Visits")
             .otherwise("Over 12 Visits")
        )
    )

    # 6) Distribution by year (denominator = cohort size)
    w = Window.partitionBy("visit_year")

    df_bin_distribution = (
        df_visits_binned
        .groupBy("visit_year", "visit_bin")
        .agg(F.countDistinct("cde_id").alias("members"))
        .withColumn("proportion", F.round(F.col("members") / F.sum("members").over(w),4))
        .orderBy("visit_year", "visit_bin")
    )

    df_year_summary = (
        df_visits_full
        .groupBy("visit_year")
        .agg(
            F.countDistinct("cde_id").alias("population"),
            F.sum("total_visits_year").alias("total_visits"),
            F.avg("total_visits_year").alias("avg_visits_per_customer_including_zeros")
        )
        .orderBy("visit_year")
    )

    df_year_summary.display()

    # 7) OUTPUT
    df_bin_distribution.display()
 

# ---------------------------------------------------------
# (A) Month signups cohort (merge_df)
# ---------------------------------------------------------
show_visit_distribution(merge_df, cohort_name="Month Signups", id_col="cde_id")

# ---------------------------------------------------------
# (B) Program Till Date cohort (ptd_signup)
# ---------------------------------------------------------
show_visit_distribution(ptd_signup, cohort_name="Program Till Date (PTD)", id_col="cde_id")

show_visit_distribution(df_scene_member_visit, cohort_name="SCENE Members", id_col="cde_id")


In [0]:
start_prev_month = F.to_date(F.concat(F.lit(MOM_month), F.lit("-01")))
end_prev_month   = F.last_day(start_prev_month)

# Window for total visits per customer (same as before)
w_cust = Window.partitionBy("cde_id")

# NEW: visit-grain table with ticket/concession separated BEFORE aggregation
df_tickets_main = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre") &
        (F.col("CineClubFLAG") == "Yes")
    )
    .withColumnRenamed("CDE_ID", "cde_id")
    .withColumn("Mth", F.date_format(F.col("datefull"), "yyyy-MM"))
    .withColumn("ticket_qty", F.when(F.col("TicketFLAG") == 'Yes', F.col("Quantity")).otherwise(F.lit(0)))
    .withColumn("concession_qty", F.when(F.col("ConcessionFLAG") == 'Yes', F.col("Quantity")).otherwise(F.lit(0)))  # only if column exists
    .groupBy("cde_id", "datefull", T_trans.LocationId, "Mth")
    .agg(
        F.sum("Quantity").alias("Net_Quantity"),
        F.sum("ticket_qty").alias("Net_Ticket_Quantity"),
        F.sum("concession_qty").alias("Net_Concession_Quantity") 
    )
    .filter((F.col("Net_Quantity") > 0) & (F.col("cde_id") > 0))
    .withColumn("Total_Visits", F.count(F.lit(1)).over(w_cust))
)

# Example: the exact metric you started with, now month-driven and ticket-correct
df_cineclub_active_members_total_tickets = (
    df_tickets_main
    .filter(F.col("datefull").between(start_prev_month, end_prev_month))
    .groupBy(F.lit(MOM_month).alias("Month"))
    .agg(
        F.countDistinct("cde_id").alias("Members_with_1_plus_visits"),
        F.sum("Net_Ticket_Quantity").alias("Total_Tickets")
    )
)


In [0]:
from pyspark.sql import functions as F, Window

# Month bounds from MOM_month = "yyyy-MM"
start_prev_month = F.to_date(F.concat(F.lit(MOM_month), F.lit("-01")))
end_prev_month   = F.last_day(start_prev_month)

# Base visit-grain table (history-wide)
df_tickets_main = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre") &
        (F.col("CineClubFLAG") == "Yes")
    )
    .withColumnRenamed("CDE_ID", "cde_id")
    .withColumn("Mth", F.date_format(F.col("datefull"), "yyyy-MM"))
    .withColumn("ticket_qty", F.when(F.col("TicketFLAG") == "Yes", F.col("Quantity")).otherwise(F.lit(0)))
    .withColumn("concession_qty", F.when(F.col("ConcessionFLAG") == "Yes", F.col("Quantity")).otherwise(F.lit(0)))
    .groupBy("cde_id", "datefull",T_trans.LocationId, "Mth")
    .agg(
        F.sum("Quantity").alias("Net_Quantity"),
        F.sum("ticket_qty").alias("Net_Ticket_Quantity"),
        F.sum("concession_qty").alias("Net_Ticket_Concession")  # optional rename; keep yours if you prefer
    )
    .filter((F.col("Net_Quantity") > 0) & (F.col("cde_id") > 0))
)

# 1) Total visit history per member (history-wide)
df_total_visit_history = (
    df_tickets_main
    .groupBy("cde_id")
    .agg(
        F.countDistinct("datefull").alias("Total_Visits_History")
    )
)

# 2) Visits within the selected month per member
df_month_visits = (
    df_tickets_main
    .filter(F.col("datefull").between(start_prev_month, end_prev_month))
    .groupBy("cde_id")
    .agg(
        F.countDistinct("datefull").alias("Visits_In_Selected_Month"),
        F.sum("Net_Ticket_Quantity").alias("Tickets_In_Selected_Month")
    )
)

# Member-level output with both metrics
df_member_visit_metrics = (
    df_total_visit_history
    .join(df_month_visits, "cde_id", "left")
    .fillna({"Visits_In_Selected_Month": 0, "Tickets_In_Selected_Month": 0})
    .withColumn("Month", F.lit(MOM_month))
)


In [0]:
start_prev_month = F.to_date(F.concat(F.lit(MOM_month), F.lit("-01")))
end_prev_month   = F.last_day(start_prev_month)

def build_visit_grain_base(cohort_flag_col: str, cohort_flag_val: str):
    """
    cohort_flag_col: e.g., "CineClubFLAG" or "SCENEFLAG"
    cohort_flag_val: e.g., "Yes"
    """
    return (
        T_trans
        .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
        .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
        .filter(
            (T_locations.LC_Location_Type_Description == "Cineplex Theatre") &
            (F.col("CDE_ID").isNotNull()) &
            (F.col(cohort_flag_col) == cohort_flag_val)
        )
        .withColumnRenamed("CDE_ID", "cde_id")
        .withColumn("Mth", F.date_format(F.col("datefull"), "yyyy-MM"))
        .withColumn("ticket_qty", F.when(F.col("TicketFLAG") == "Yes", F.col("Quantity")).otherwise(F.lit(0)))
        .withColumn("concession_qty", F.when(F.col("ConcessionFLAG") == "Yes", F.col("Quantity")).otherwise(F.lit(0)))
        .groupBy("cde_id", "datefull", T_trans.LocationId, "Mth")
        .agg(
            F.sum("Quantity").alias("Net_Quantity"),
            F.sum("ticket_qty").alias("Net_Ticket_Quantity"),
            F.sum("concession_qty").alias("Net_Concession_Quantity")
        )
        .filter((F.col("Net_Quantity") > 0) & (F.col("cde_id") > 0))
    )

def compute_month_summary(df_visit_grain_base, cohort_label: str):
    # 2) Member-level: total visit history
    df_total_visit_history = (
        df_visit_grain_base
        .groupBy("cde_id")
        .agg(F.countDistinct("datefull").alias("Total_Visits_History"))
    )

    # 3) Member-level: selected month visits + tickets
    df_month_member_metrics = (
        df_visit_grain_base
        .filter(F.col("datefull").between(start_prev_month, end_prev_month))
        .groupBy("cde_id")
        .agg(
            F.countDistinct("datefull").alias("Visits_In_Selected_Month"),
            F.sum("Net_Ticket_Quantity").alias("Tickets_In_Selected_Month")
        )
    )

    # 4) Final member table (history + selected month)
    df_member_visit_metrics = (
        df_total_visit_history
        .join(df_month_member_metrics, "cde_id", "left")
        .fillna({"Visits_In_Selected_Month": 0, "Tickets_In_Selected_Month": 0})
        .withColumn("Month", F.lit(MOM_month))
        .withColumn("Cohort", F.lit(cohort_label))
    )

    # 5) Month-level summary
    df_month_summary = (
        df_member_visit_metrics
        .groupBy("Month", "Cohort")
        .agg(
            F.countDistinct(F.when(F.col("Visits_In_Selected_Month") > 0, F.col("cde_id"))).alias("Members_with_1_plus_visits"),
            F.sum("Tickets_In_Selected_Month").alias("Total_Tickets_In_Month"),
            F.sum("Visits_In_Selected_Month").alias("Total_Visits_In_Month"),
            F.sum("Total_Visits_History").alias("Total_Visits_History_Summed")
        )
    )

    return df_month_summary, df_member_visit_metrics

df_tickets_main_cc    = build_visit_grain_base("CineClubFLAG", "Yes")
df_tickets_main_scene = build_visit_grain_base("SCENEFLAG", "Yes")

df_cc_summary, df_cc_member_metrics       = compute_month_summary(df_tickets_main_cc, "CineClub")
df_scene_summary, df_scene_member_metrics = compute_month_summary(df_tickets_main_scene, "SCENE")

# Combined output (two rows: CineClub + SCENE for the same MOM_month)
df_month_summary_all = df_cc_summary.unionByName(df_scene_summary)
df_month_summary_all.display()


In [0]:
from pyspark.sql import functions as F

# ============================================================
# 0) Month window (MOM_month in "YYYY-MM")
# ============================================================
# Example: MOM_month = "2025-12"
start_prev_month = F.to_date(F.concat(F.lit(MOM_month), F.lit("-01")))
end_prev_month   = F.last_day(start_prev_month)

# ============================================================
# 1) Cineplex format mapping + classifier
# ============================================================
top_tier_mapping = {
    2: "AVX",
    3: "IMAX",
    4: "VIP",
    5: "ALT",
    6: "ALT",
    7: "FourDX",
    8: "ScreenX"
}
mapping_expr = F.create_map([F.lit(x) for x in sum(top_tier_mapping.items(), ())])

def add_ticket_format(df_in):
    return (
        df_in
        .withColumn("TopTier_Label", mapping_expr.getItem(F.col("Meta_Auditorium_Experience_ID")))
        .withColumn(
            "Final_Format",
            F.when(F.col("TopTier_Label").isNotNull(), F.col("TopTier_Label"))
             .when(((F.col("Meta_Performance_Experience_ID") == 2)), "ThreeD")
             .otherwise("Regular")
        )
    )

# ============================================================
# 2) TABLE DEFINITIONS (MoM-filtered, refund-net ticket-event grain)
#    Grain: cde_id × datefull × LocationId × sessionskey × TicketTypeSkey
#    Purpose: ticket format classification & ticket counts (not deduped)
# ============================================================

# ----------------------------
# 2A) CineClub ticket-event table
# ----------------------------
df_tickets_main_cc = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_ticket, T_trans.TicketTypeSkey == T_ticket.tickettypeskey, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre") &
        (F.col("CineClubFLAG") == "Yes") &
        (F.col("TicketFLAG") == "Yes") &
        (F.col("CDE_ID").isNotNull()) &
        (F.col("datefull").between(start_prev_month, end_prev_month))
    )
    .select(
        F.col("CDE_ID").alias("cde_id"),
        "datefull",
        T_trans.LocationId.alias("LocationId"),
        "sessionskey",
        T_trans.TicketTypeSkey,
        "Quantity"
    )
    .groupBy("cde_id", "datefull", "LocationId", "sessionskey", T_trans.TicketTypeSkey)
    .agg(F.sum("Quantity").alias("Net_Ticket_Qty"))
    .filter((F.col("Net_Ticket_Qty") > 0) & (F.col("cde_id") > 0))
)

# ----------------------------
# 2B) SCENE ticket-event table
# ----------------------------
df_tickets_main_totalbusiness = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_ticket, T_trans.TicketTypeSkey == T_ticket.tickettypeskey, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre") &
        (F.col("TicketFLAG") == "Yes") &
        (F.col("datefull").between(start_prev_month, end_prev_month))
    )
    .select(
        F.col("CDE_ID").alias("cde_id"),
        "datefull",
        T_trans.LocationId.alias("LocationId"),
        "sessionskey",
        T_trans.TicketTypeSkey,
        "Quantity"
    )
    .groupBy("cde_id", "datefull", "LocationId", "sessionskey", T_trans.TicketTypeSkey)
    .agg(F.sum("Quantity").alias("Net_Ticket_Qty"))
    .filter((F.col("Net_Ticket_Qty") > 0))
)

# ============================================================
# 3) Attach session metadata + classify format
# ============================================================
def attach_meta_and_classify(df_ticket_event):
    df_meta = (
        df_ticket_event
        .join(
            T_sessions.select(
                "sessionskey",
                "Auditorium_Experience_ID",
                "Performance_Experience_ID",
                "Is_DBOXflag"
            ),
            "sessionskey",
            "left"
        )
        .withColumnRenamed("Auditorium_Experience_ID", "Meta_Auditorium_Experience_ID")
        .withColumnRenamed("Performance_Experience_ID", "Meta_Performance_Experience_ID")
    )
    return add_ticket_format(df_meta)

df_cc_with_format    = attach_meta_and_classify(df_tickets_main_cc)
df_totalbusiness_with_format = attach_meta_and_classify(df_tickets_main_totalbusiness)

# ============================================================
# 4) Aggregate: net ticket counts by format (+ % share)
# ============================================================
def format_ticket_summary(df_with_format):
    summary = (
        df_with_format
        .groupBy("Final_Format")
        .agg(
            F.sum("Net_Ticket_Qty").alias("Tickets_Sold_Net"),
            F.countDistinct("cde_id").alias("Unique_Buyers")
        )
    )

    total_row = summary.agg(F.sum("Tickets_Sold_Net").alias("t")).collect()
    total = total_row[0]["t"] if len(total_row) > 0 and total_row[0]["t"] is not None else 0

    return (
        summary
        .withColumn(
            "Pct_of_Tickets",
            F.when(F.lit(total) > 0, F.round(F.col("Tickets_Sold_Net") / F.lit(total) * 100, 2))
             .otherwise(F.lit(0.0))
        )
        .orderBy(F.desc("Tickets_Sold_Net"))
    )

df_cc_format_ticket_counts    = format_ticket_summary(df_cc_with_format)
df_totalbusiness_format_ticket_counts = format_ticket_summary(df_totalbusiness_with_format)

# ============================================================
# 5) Outputs
# ============================================================
display(Markdown("**Ticket Format - CC**"))
df_cc_format_ticket_counts.display()
df_cc_format_ticket_counts.agg(F.sum("Tickets_Sold_Net").alias("Total_Tickets_Sold")).show()
display(Markdown("**Ticket Format - Total Business**"))
df_totalbusiness_format_ticket_counts.display()
df_totalbusiness_format_ticket_counts.agg(F.sum("Tickets_Sold_Net").alias("Total_Tickets_Sold")).show()


In [0]:
%sql
SELECT   CineClub_Discount_Type_ID
          ,month(cast (Transaction_DateTime as date)) as mm
        ,count (distinct transaction_number) as tot_trans
    ,SUM(Quantity) as Total_items
    ,SUM(Net_Spend) as Net_Spend
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
  WHERE  CineClub_Discount_Type_ID in (2,3)
  and year(cast (Transaction_DateTime as date)) = 2026
    group by CineClub_Discount_Type_ID,
         month(cast (Transaction_DateTime as date))

In [0]:
%sql
----- CineClub Q3
 
SELECT COUNT(DISTINCT t.CDE_ID)
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_location s
    ON s.LocationID = t.LocationId
LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
    ON d.dateid = t.VisitDateId
WHERE t.CineClubFLAG = 'Yes'
  AND d.datefull BETWEEN DATE '2026-01-01' AND DATE '2026-02-28'
  AND s.LC_Location_Type_Description = 'Cineplex Theatre';

 
------- Scene+ Q3
 
SELECT COUNT(DISTINCT t.CDE_ID)
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_location s
    ON s.LocationID = t.LocationId
LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
    ON d.dateid = t.VisitDateId
WHERE t.SCENEFLAG = 'Yes'
  AND d.datefull BETWEEN DATE '2026-01-01' AND DATE '2026-02-28'
  AND s.LC_Location_Type_Description = 'Cineplex Theatre';


## P&L Concession Trend

In [0]:
# df_conc (your existing logic)
df_conc_ = (
    T_trans
    .join(T_date, F.col("VisitDateId") == F.col("dateid"), "left")
    .filter(
        (F.col("CineClubFLAG") == "Yes") &
        (F.date_format("datefull", "yyyy-MM") == MOM_month) &
        (F.col("CineClub_Discount_Type_ID") == 2)
    )
)

# 1-row summary
df_conc_summary = (
    df_conc_.agg(
        F.count("*").alias("Total_Transactions"),
        F.sum(F.col("Quantity").cast("double")).alias("Total_Items"),
        F.sum(F.col("Net_Revenue").cast("double")).alias("Net_Amount")
    )
    .withColumn("Discount_Amount", F.col("Net_Amount")/4)
)

df_conc_summary.display()

In [0]:
# df_conc (your existing logic)
df_conc = (
    T_trans
    .join(T_date, F.col("VisitDateId") == F.col("dateid"), "left")
    .filter(
        (F.col("CineClubFLAG") == "Yes") &
         (F.date_format("datefull", "yyyy-MM") == MOM_month) &
        (F.col("CineClub_Discount_Type_ID") == 3)
    )
)

# 1-row summary
df_conc_summary = (
    df_conc.agg(
        F.count("*").alias("Total_Transactions"),
        F.sum(F.col("Quantity").cast("double")).alias("Total_Items"),
        F.sum(F.col("Net_Revenue").cast("double")).alias("Net_Amount")
    )
    .withColumn("Discount_Amount", F.col("Net_Amount")/4)
)

df_conc_summary.display()

## CineClub Concession Trend

In [0]:
from pyspark.sql import functions as F

df_Conc_Trend = (
    T_trans
    .join(T_date, F.col("VisitDateId") == F.col("dateid"), "left")
    .join(
        T_locations.select("LocationID", "LC_Location_Type_Description"),
        T_trans.LocationId == T_locations.LocationID,
        "left"
    )
    .filter(F.date_format(F.col("datefull"), "yyyy-MM") == MOM_month)
    .filter(F.col("RecRoomFLAG") == "No")
    .filter(F.col("STOREFLAG") == "No")
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .groupBy("CDE_ID", T_trans.LocationId, "datefull", "Transaction_Number")
    .agg(
        # Flags
        F.max(F.when(F.col("ConcessionFLAG") == "Yes", 1).otherwise(0)).alias("Has_Concession"),
        F.max(F.when(F.col("CineClubFLAG") == "Yes", 1).otherwise(0)).alias("Is_CineClub_Transaction"),
        F.max(
            F.when((F.col("CineClubFLAG") == "Yes") & (F.col("ConcessionFLAG") == "Yes"), 1).otherwise(0)
        ).alias("Is_CineClub_Concession_Transaction"),

        # ✅ Total Concession Revenue (ALL concession lines)
        F.sum(
            F.when(F.col("ConcessionFLAG") == "Yes", F.col("Net_Revenue").cast("double")).otherwise(0)
        ).alias("Concession_Revenue"),

        # ✅ Optional: Concession Revenue where discount type = 2 (your new condition)
        F.sum(
            F.when(
                (F.col("ConcessionFLAG") == "Yes") &
                (F.col("CineClub_Discount_Type_ID").cast("int") == 2),
                F.col("Net_Revenue").cast("double")
            ).otherwise(0)
        ).alias("Concession_Revenue_CCType2"),

        # Attendance
        F.sum(F.when(F.col("TicketFLAG") == "Yes", F.col("Quantity").cast("double")).otherwise(0))
            .alias("Attendance"),

        # CineClub Attendance
        F.sum(
            F.when(
                (F.col("TicketFLAG") == "Yes") & (F.col("CineClubFLAG") == "Yes"),
                F.col("Quantity").cast("double")
            ).otherwise(0)
        ).alias("CineClub_Attendance")
    )
    .withColumn(
        "Is_NonCineClub_Concession_Transaction",
        F.when((F.col("Has_Concession") == 1) & (F.col("Is_CineClub_Transaction") == 0), 1).otherwise(0)
    )
    .drop("Has_Concession")
)

df_totals = (
    df_Conc_Trend.agg(
        F.sum("Is_CineClub_Concession_Transaction").alias("Total_CineClub_Concession_Transactions"),
        F.sum("Is_NonCineClub_Concession_Transaction").alias("Total_NonCineClub_Concession_Transactions"),
        F.sum("CineClub_Attendance").alias("Total_CineClub_Attendance"),
        F.sum("Attendance").alias("Total_Attendance"),
        F.sum("Concession_Revenue").alias("Total_Concession_Revenue"),
        F.sum("Concession_Revenue_CCType2").alias("Total_Concession_Revenue_CC")
    )
    .withColumn(
        "Total_Transactions",
        F.col("Total_CineClub_Concession_Transactions") + F.col("Total_NonCineClub_Concession_Transactions")
    )
)


In [0]:
df_totals.display()

## 9.99 Tickets


In [0]:
df_999_Ticket = (
    T_trans
    .join(T_date, F.col("VisitDateId") == F.col("dateid"), "left")
    .join(
        T_locations, T_trans.LocationId == T_locations.LocationID, "left"
    )
    .filter(F.date_format(F.col("datefull"), "yyyy-MM") == MOM_month)
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .filter(F.col("CineClub_Discount_Type_ID").isin(5,6))
    .groupBy("CDE_ID", "datefull", T_trans.LocationId)
    .agg(
        F.sum(F.col("Quantity")).alias("Total 999 Tickets")
    )
)

df_999_Ticket.agg(F.sum("Total 999 Tickets")).display()